# HDEM Static — Ablation Study (No Dynamic Weight Update)

This notebook trains HDEM **without** the sliding window dynamic weight update mechanism.
Sub-ensemble weights remain at their initial uniform values (1/K).

Compare results with `HDEM_train.ipynb` to measure the contribution of dynamic weight adaptation.

In [2]:
import os
import sys
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
import time
from preprocessing import *
from HDEM_static import Static_Weighted_Ensemble

# set_seed(42)

In [3]:
df = pd.read_csv(r'../output_csv/HCMUT-SuperNodeXP-2017-1.0.swf.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15886 entries, 0 to 15885
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   job_id                    15886 non-null  float64
 1   submit_time               15886 non-null  float64
 2   wait_time                 15886 non-null  float64
 3   run_time                  15886 non-null  float64
 4   num_allocated_processors  15886 non-null  float64
 5   avg_cpu_time_used         15886 non-null  float64
 6   used_memory               15886 non-null  float64
 7   requested_processors      15886 non-null  float64
 8   requested_time            15886 non-null  float64
 9   requested_memory          15886 non-null  float64
 10  status                    15886 non-null  float64
 11  user_id                   15886 non-null  float64
 12  group_id                  15886 non-null  float64
 13  executable_id             15886 non-null  float64
 14  queue_

In [4]:
feature_columns = ['requested_processors', 'requested_time', 'avg_cpu_time_used', 'used_memory', 'submit_time', 'wait_time', 'user_id', 'group_id', 'executable_id', 'queue_id']
target_column = 'run_time'

X_train, X_val, X_test, Y_train, Y_val, Y_test, scaler = prepare_data_DL(df, feature_columns, target_column)

In [5]:
rmse_lst, mae_lst, r2_lst, infer_time_lst = [], [], [], []
for iter in range(5):
    print(f"Iteration {iter + 1} / 5")
    HDEM_static = Static_Weighted_Ensemble(X_train, X_val, X_test, Y_train, Y_val, Y_test, scaler)

    HDEM_static.meta_model_name = 'gradientboosting'
    model_combinations = [['extratrees', 'randomforest', 'xgboost'], ['randomforest', 'mlp', 'gradientboosting'], ['lasso', 'xgboost', 'extratrees']]
    HDEM_static.num_sub = len(model_combinations)
    HDEM_static.init_base_sub(model_combinations)
    sub_ensemble_result = HDEM_static.run_model()
    rmse, mae, r2, infer_time = sub_ensemble_result['RMSE'], sub_ensemble_result['MAE'], sub_ensemble_result['R² Score'], sub_ensemble_result['Inference Time']
    rmse_lst.append(rmse)
    mae_lst.append(mae)
    r2_lst.append(r2)
    infer_time_lst.append(infer_time)

Iteration 1 / 5


Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  10527.589739458232
RMSE:  30396.737462365458
R2:  0.9413452235599313
Inference Time:  3.396355828573538e-05
Iteration 2 / 5
Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  10583.414532825931
RMSE:  29617.16938206444
R2:  0.9443152162960807
Inference Time:  3.0157177947288336e-05
Iteration 3 / 5
Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  10526.62007388465
RMSE:  29733.18178793812
R2:  0.9438781199609229
Inference Time:  3.298424011053041e-05
Iteration 4 / 5
Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  10142.997060956119
RMSE:  29188.38925724865
R2:  0.9459158887280682
Inference Time:  2.64221845671188e-05
Iteration 5 / 5
Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  10789.061326951298
RMSE:  30893.255002502778
R2:  0.9394133726716333
Inference Time:  2.9759351597275844e-05


In [ ]:
# Save results to CSV
results_df = pd.DataFrame({
    'RMSE': rmse_lst,
    'MAE': mae_lst,
    'R2': r2_lst,
    'Inference Time': infer_time_lst
})
results_df.to_csv('../output_HCMUT/HDEM_static_results.csv', index=False)

print(f"\n{'='*60}")
print(f"HDEM Static (No Dynamic Weight) — HCMUT Dataset")
print(f"{'='*60}")
print(f"Mean RMSE: {np.mean(rmse_lst):.4f} ± {np.std(rmse_lst):.4f}")
print(f"Mean MAE:  {np.mean(mae_lst):.4f} ± {np.std(mae_lst):.4f}")
print(f"Mean R²:   {np.mean(r2_lst):.4f} ± {np.std(r2_lst):.4f}")
print(f"Mean Inference Time: {np.mean(infer_time_lst):.6f}s")


HDEM Static (No Dynamic Weight) — HCMUT Dataset
Mean RMSE: 29965.7466 ± 604.3016
Mean MAE:  10513.9365 ± 209.0283
Mean R²:   0.9430 ± 0.0023
Mean Inference Time: 0.000031s


: 